In [ ]:
# CELL 1: Initial Setup
# Mount Google Drive and import libraries
from google.colab import drive
drive.mount('/content/drive')

!pip install boto3 awscli
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import boto3
from tensorflow.keras import layers, models, applications, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# === GPU CONFIGURATION ===
# Check for GPU
print("TensorFlow version:", tf.__version__)
physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))
for device in physical_devices:
    print(f"GPU device: {device}")

# Configure TensorFlow to use GPU memory efficiently
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Set TensorFlow to only use a portion of GPU memory as needed
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")

# Enable mixed precision training
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print('Compute dtype:', policy.compute_dtype)
print('Variable dtype:', policy.variable_dtype)

# Add project root to Python path
project_root = '/content/drive/MyDrive/Colab Notebooks/Pawnder'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# === AWS CONFIGURATION ===
# Try to get AWS credentials from environment or Colab secrets
try:
    from google.colab import userdata
    aws_access_key = userdata.get('AWS_ACCESS_KEY_ID')
    aws_secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')
    aws_region = userdata.get('AWS_REGION', 'us-east-1')

    # Set environment variables for AWS SDK
    os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_key
    os.environ['AWS_DEFAULT_REGION'] = aws_region

    # Initialize S3 client
    s3 = boto3.client('s3')
    aws_connected = True
    print("AWS credentials loaded from Colab secrets")
except Exception as e:
    print(f"AWS configuration not available: {str(e)}")
    aws_connected = False


In [ ]:
# CELL 2: Monitor GPU usage
!nvidia-smi

In [ ]:
# CELL 3: Copy data from Google Drive to local storage with progress
!mkdir -p /content/dataset
!apt-get install -y rsync
!rsync -av --progress "/content/drive/MyDrive/Colab Notebooks/Pawnder/Data/processed/" "/content/dataset/"

In [ ]:
# CELL 4: Configuration for model training
# === MODEL CONFIGURATION ===
MODEL_NAME = "pawnder_behavior_v1"
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
EPOCHS = 50
LEARNING_RATE = 0.0001
PATIENCE = 10  # For early stopping

# Define the emotion classes exactly as they appear in folder names
CLASS_NAMES = ["Happy_Playful", "Relaxed", "Submissive_Appeasement",
               "Curiosity_Alertness", "Stressed", "Fearful_Anxious",
               "Aggressive_Threatening"]

# Define data paths
# BASE_DIR = "/content/dataset"  # Local directory
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Pawnder/Data/processed"  # Original Google Drive path

# Define paths
train_dir = os.path.join(BASE_DIR, 'train_by_class')
valid_dir = os.path.join(BASE_DIR, 'val_by_class')
test_dir = os.path.join(BASE_DIR, 'test_by_class')

# Create output directories for model artifacts
# If DATA_DIRS doesn't have 'models' key or isn't defined
if 'DATA_DIRS' not in globals() or 'models' not in DATA_DIRS:
    DATA_DIRS = {} if 'DATA_DIRS' not in globals() else DATA_DIRS
    DATA_DIRS['models'] = os.path.join(project_root, 'Models')
    os.makedirs(DATA_DIRS['models'], exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
model_dir = os.path.join(DATA_DIRS['models'], MODEL_NAME, timestamp)
checkpoints_dir = os.path.join(model_dir, 'checkpoints')
plots_dir = os.path.join(model_dir, 'plots')
results_dir = os.path.join(model_dir, 'results')

# Create the directories if they don't exist
os.makedirs(model_dir, exist_ok=True)
os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print("Training directories created:")
print(f"  Model dir: {model_dir}")
print(f"  Checkpoints dir: {checkpoints_dir}")

In [ ]:
import tensorflow as tf

# Define these variables if not already defined
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

AUTOTUNE = tf.data.AUTOTUNE

def create_dataset(directory, batch_size):
    # Load the raw dataset first
    raw_dataset = tf.keras.preprocessing.image_dataset_from_directory(
        directory,
        image_size=IMAGE_SIZE,
        batch_size=batch_size,
        label_mode='categorical'
    )

    # Store the class names before applying transformations
    class_names = raw_dataset.class_names

    # Apply the transformations
    dataset = raw_dataset.cache().prefetch(buffer_size=AUTOTUNE)

    # Return both the transformed dataset and the class names
    return dataset, class_names

# Create datasets with caching and prefetching for better performance
train_dataset, train_class_names = create_dataset(train_dir, BATCH_SIZE)
val_dataset, val_class_names = create_dataset(valid_dir, BATCH_SIZE)
test_dataset, test_class_names = create_dataset(test_dir, BATCH_SIZE)

# Print the actual classes detected from the directories
print(f"Classes detected: {train_class_names}")
print(f"Number of training batches: {len(train_dataset)}")
print(f"Number of validation batches: {len(val_dataset)}")
print(f"Number of test batches: {len(test_dataset)}")

In [ ]:
# CELL 6: Create and train the model
# Create data generators with augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2]  # Add brightness variation
)

# Only rescaling for validation and test
valid_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Calculate class weights based on the inverse of frequencies
class_weights = {
    0: 1.0,                    # Happy_Playful
    1: 1.65,                   # Relaxed
    2: 9.8,                    # Submissive_Appeasement
    3: 12.8,                   # Curiosity_Alertness
    4: 20.8,                   # Stressed
    5: 0.96,                   # Fearful_Anxious
    6: 8.1                     # Aggressive_Threatening
}

# Create a function to build the model
def build_model(num_classes=len(CLASS_NAMES)):
    """Build model using transfer learning"""
    # Use MobileNetV2 as base model (efficient for mobile deployment)
    base_model = applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )

    # Freeze the base model layers
    base_model.trainable = False

    # Build model on top of the base model
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')  # Outputs matching classes
    ])

    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Build the model
model = build_model()
model.summary()

# Set up callbacks for training
callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        filepath=os.path.join(checkpoints_dir, 'model-{epoch:02d}-{val_accuracy:.4f}.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    ),
    callbacks.TensorBoard(
        log_dir=os.path.join(model_dir, 'logs')
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001
    )
]

# Train the model with class weights to handle imbalance
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks_list,
    class_weight=class_weights  # Use class weights
)

# Save final model
model.save(os.path.join(model_dir, f'{MODEL_NAME}_final.h5'))
print(f"Model saved to: {os.path.join(model_dir, f'{MODEL_NAME}_final.h5')}")

In [ ]:
# 8. Visualize training
plot_training_history(history, model_dirs)

In [ ]:
# 9. Evaluate model
eval_results = evaluate_model(model, datasets, model_dirs)
print(f"Test accuracy: {eval_results['test_accuracy']:.4f}")

In [ ]:
# 10. Fine-tune if needed
if eval_results['test_accuracy'] > 0.6:
    fine_tune_history = fine_tune_model(model, datasets, model_dirs)
    plot_training_history(fine_tune_history, model_dirs)
    eval_results = evaluate_model(model, datasets, model_dirs)
    print(f"Fine-tuned test accuracy: {eval_results['test_accuracy']:.4f}")

In [ ]:
# 11. Sync to AWS S3
sync_results_to_s3(model_dirs)

print("Training complete!")

In [ ]:
# CELL 7: Convert model to TensorFlow Lite with robust error handling
print("Converting model to TensorFlow Lite format...")

# First attempt with standard conversion
try:
    # Create a TFLite converter from the saved Keras model
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Add all supported ops to ensure compatibility
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,  # Standard TFLite ops
        tf.lite.OpsSet.SELECT_TF_OPS     # Allow regular TensorFlow ops
    ]

    # Set optimization flags
    converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Enable default optimizations

    # Allow custom operations if needed
    converter.allow_custom_ops = True

    # Convert the model
    print("Starting conversion...")
    tflite_model = converter.convert()

    # Save the TFLite model to a file
    tflite_model_path = os.path.join(model_dir, f'{MODEL_NAME}_model.tflite')
    with open(tflite_model_path, 'wb') as f:
        f.write(tflite_model)
    print(f"TensorFlow Lite model saved to: {tflite_model_path}")
    print(f"TFLite model size: {os.path.getsize(tflite_model_path) / (1024 * 1024):.2f} MB")

    # Try to create a quantized version if the first conversion succeeded
    try:
        # Create a quantized version for smaller size
        converter_quantized = tf.lite.TFLiteConverter.from_keras_model(model)
        converter_quantized.optimizations = [tf.lite.Optimize.DEFAULT]
        converter_quantized.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS
        ]
        converter_quantized.target_spec.supported_types = [tf.float16]
        converter_quantized.allow_custom_ops = True

        tflite_model_quantized = converter_quantized.convert()

        # Save the quantized model
        tflite_quantized_path = os.path.join(model_dir, f'{MODEL_NAME}_quantized.tflite')
        with open(tflite_quantized_path, 'wb') as f:
            f.write(tflite_model_quantized)
        print(f"Quantized TFLite model saved to: {tflite_quantized_path}")
        print(f"Quantized TFLite model size: {os.path.getsize(tflite_quantized_path) / (1024 * 1024):.2f} MB")
    except Exception as quantize_error:
        print(f"Warning: Quantization failed with error: {str(quantize_error)}")
        print("Skipping quantized model generation.")

except Exception as main_error:
    print(f"Standard conversion failed with error: {str(main_error)}")
    print("Attempting alternative conversion approach...")

    try:
        # Try a different approach using a concrete function
        run_model = tf.function(lambda x: model(x))
        concrete_func = run_model.get_concrete_function(
            tf.TensorSpec([1, 224, 224, 3], tf.float32))  # Adjust shape if needed

        converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS
        ]
        converter.allow_custom_ops = True

        tflite_model = converter.convert()

        # Save the alternative TFLite model
        tflite_model_path = os.path.join(model_dir, f'{MODEL_NAME}_model_alt.tflite')
        with open(tflite_model_path, 'wb') as f:
            f.write(tflite_model)
        print(f"Alternative TFLite model saved to: {tflite_model_path}")
        print(f"Alternative TFLite model size: {os.path.getsize(tflite_model_path) / (1024 * 1024):.2f} MB")

    except Exception as alt_error:
        print(f"Alternative conversion also failed: {str(alt_error)}")
        print("\nTroubleshooting suggestions:")
        print("1. Try saving model as a SavedModel format first")
        print("2. Check model architecture for compatibility with TFLite")
        print("3. Try removing dropout or batch normalization layers if present")
        print("4. Consider using a simpler model architecture for mobile deployment")

finally:
    # Always save class names regardless of conversion success
    class_names_path = os.path.join(model_dir, 'class_names.json')
    with open(class_names_path, 'w') as f:
        json.dump({"class_names": CLASS_NAMES}, f)
    print(f"Class names saved to: {class_names_path}")